# Lab 5 · Notebook 01 – Chuẩn bị nhãn và dữ liệu

**Bài toán:** phân loại mức **rủi ro PM2.5 minh họa** từ các cảm biến phụ trợ. Nhãn nhị phân dựa trên trung bình PM2.5 của 24 giờ gần nhất, ngưỡng 75 µg/m³ chỉ dùng **cho bài tập**, **không phải AQI chính thức** hay cảnh báo sức khỏe thực tế.

**Không được** đưa PM2.5, trung bình trượt PM2.5 hay nhãn gốc vào X. Dữ liệu mặc định **giả lập**.

In [ ]:
from pathlib import Path
import sys
ROOT = next((p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
             if (p / "src" / "lab5.py").exists()), None)
assert ROOT, "Hãy chạy Jupyter trong repository FIT4112 Lab 5"
if str(ROOT) not in sys.path: sys.path.insert(0,str(ROOT))
print("Repo:", ROOT)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from src.lab5 import load_data,prepare_frame,FEATURES,TARGET,THRESHOLD

DATA_SOURCE="demo"    # hoặc "uci_zip" (nâng cao)
STATION="Aotizhongxin"
raw=load_data(ROOT,source=DATA_SOURCE,station=STATION)
print("Records:",len(raw),"| missing:",raw.isna().sum().to_dict())
frame=prepare_frame(raw)
print("Features (allowed):",FEATURES)
display(frame.head())
print("Class counts:",frame[TARGET].value_counts().to_dict())
assert all(c not in FEATURES for c in ("PM2.5","pm25_24h","risk_label","datetime"))

### Khám phá dữ liệu

Xem phân bố nhãn và một số đặc trưng. Nhãn được xác định từ PM2.5 nhưng **tất cả phép huấn luyện chỉ đọc tập FEATURES được phép**, không dùng nhãn ẩn hay thuộc tính dùng trực tiếp để tạo nhãn.

In [ ]:
frame[TARGET].value_counts().sort_index().plot(kind="bar",title="Educational PM2.5-risk proxy (0=low, 1=high)")
plt.xlabel("Risk proxy");plt.ylabel("Count");plt.tight_layout();plt.show()
fig,ax=plt.subplots(figsize=(10,3))
ax.plot(frame["datetime"].iloc[:168],frame["pm25_24h"].iloc[:168],label="Rolling PM2.5 (LABEL CONSTRUCTION ONLY)")
ax.axhline(THRESHOLD,color="grey",linestyle="--",label="Teaching threshold")
ax.legend();plt.xticks(rotation=20);plt.tight_layout();plt.show()
out=ROOT/"data/processed";out.mkdir(parents=True,exist_ok=True)
frame.to_csv(out/"prepared_demo_or_uci.csv",index=False)
print("Saved:",out/"prepared_demo_or_uci.csv")

### Câu hỏi
1. Nhãn 0/1 được tạo như thế nào? Vì sao không được gọi là AQI chính thức?
2. Vì sao loại PM2.5 và pm25_24h khỏi features, dù chúng rất liên quan đến nhãn?
3. Hai lớp có cân bằng không? Dữ liệu giả lập có thể dùng để cảnh báo thật không?